# Chargement du modèle et du tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, EarlyStoppingCallback

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = model

Loading weights: 100%|██████████| 338/338 [00:03<00:00, 97.58it/s]


# Configuration de LoRA

In [ ]:
from peft import LoraConfig

# Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 8
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 16
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

# Chargement des données

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "../datasets/train.json",
    "validation": "../datasets/validation.json"
}, field="examples")

def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False, 
        add_generation_prompt=False
    )
    return example

dataset = dataset.map(apply_chat_template)

# Préparation de l'entraînement

In [ ]:
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
import wandb

USE_CUDA = torch.cuda.is_available()
USE_BF16 = USE_CUDA and torch.cuda.is_bf16_supported()

training_args = SFTConfig(
    output_dir="../model_adapter/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    max_length=512,  # longueur max des séquences
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,  # garde le meilleur checkpoint
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=USE_BF16,
    use_cpu=not USE_CUDA,
    report_to="wandb",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    peft_config=peft_config,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)

# Entraînement

In [ ]:
wandb.init(project="Mini-Projet-GameMaster", name="QLoRA-Run-1")

In [ ]:
print("Début de l'entraînement...")
trainer.train()

print("Sauvegarde du modèle final dans model_adapter...")
trainer.model.save_pretrained("../model_adapter/best_gm_adapter")
tokenizer.save_pretrained("../model_adapter/best_gm_adapter")

wandb.finish()

# Chargement du nouveau model

In [ ]:
from peft import PeftModel

peft_model = PeftModel.from_pretrained(
    base_model, "../model_adapter/best_gm_adapter", torch_dtype=torch.float16
)

# 3. Merge adapter weights with base model
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained("../models/merged_model")
tokenizer.save_pretrained("../models/merged_model")

Test

In [ ]:
messages = [
    {
        "role": "system", 
        "content": "Tu es le Game Master d'un jeu de rôle fantasy en solo. Tu proposes trois actions au joueur. Le joueur peut reformuler une action avec ses propres mots, mais une action qui ne correspond à aucune possibilité proposée doit être refusée."
    },
    {
        "role": "assistant", 
        "content": "Le grand dragon noir déploie ses ailes gigantesques et bloque l'entrée de la caverne en crachant des flammes. Vous pouvez:\nChoix A : Frapper ses écailles d'un coup d'épée bien placé.\nChoix B : Utiliser un bouclier pour parer le souffle ardent.\nChoix C : Chercher une issue secondaire sur les hauteurs."
    },
    {
        "role": "user", 
        "content": "Je sors mon fusil d'assaut M4 pour l'abattre."
    }
]

# 4. Formater et générer la réponse avec tes hyperparamètres de génération
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")

# max_new_tokens=250, top_p=0.8 et temperature=1.0 comme tu l'as défini
outputs = merged_model.generate(**inputs, max_new_tokens=250, temperature=1.0, top_p=0.8)

# Affichage du résultat en ignorant le prompt de départ
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("\n--- Réponse du Game Master ---")
print(response)